#**หมายเหตุ** ข้อมูลทั้งหมด Run ผ่าน Jupyter lab

In [ ]:
import ee
import geemap

In [ ]:
ee.Authenticate()  # Prompts a browser window to log in to Google
ee.Initialize(project='capstone-491411')  # Initialize the Earth Engine API
Map = geemap.Map()
Map

**ดึงภาพ Sentinel-2**

In [ ]:
# ROI
gulf_of_THA = ee.FeatureCollection("projects/capstone-491411/assets/gulf_of_THA_bound_East1")

# Parameters
iniDate = '2025-01-01'
endDate = '2025-04-30'
cloudPerc = 30

MSI = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
SRP = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")
ozone = ee.ImageCollection('TOMS/MERGED')
pi = ee.Image.constant(3.141592)

# ---------------------------
# Water mask
# ---------------------------
forMask = (SRP.filterBounds(gulf_of_THA)
           .select('SR_B6')
           .filterMetadata('CLOUD_COVER', "less_than", 30)
           .filterDate('2025-01-01', '2025-12-31'))

mask = forMask.median().lt(10000)
mask = mask.updateMask(mask)

# ---------------------------
# Sentinel-2 filter
# ---------------------------
FC = (MSI.filterDate(iniDate, endDate)
      .filterBounds(gulf_of_THA)
      .filterMetadata('CLOUDY_PIXEL_PERCENTAGE', "less_than", cloudPerc))

# ---------------------------
# FUNCTIONS
# ---------------------------
def s2Correction(img):

    bands = ['B1','B2','B3','B4','B5','B6','B7','B8','B8A','B11','B12']

    img = ee.Image(img)

    rescale = img.select(bands).multiply(0.0001).multiply(mask)

    footprint = img.geometry()

    DEM = ee.Image('USGS/SRTMGL1_003').clip(footprint)

    DU = ozone.filterDate(iniDate, endDate).filterBounds(footprint).mean()

    imgDate = ee.Date(img.get('system:time_start'))
    FOY = ee.Date.fromYMD(imgDate.get('year'), 1, 1)
    JD = imgDate.difference(FOY, 'day').add(1)

    myCos = ee.Image(0.0172).multiply(JD.subtract(2)).cos().pow(2)
    cosd = myCos.multiply(pi.divide(180)).cos()

    d = ee.Image(1).subtract(ee.Image(0.01673)).multiply(cosd)

    SunAz = ee.Image.constant(img.get('MEAN_SOLAR_AZIMUTH_ANGLE'))
    SunZe = ee.Image.constant(img.get('MEAN_SOLAR_ZENITH_ANGLE'))

    cosdSunZe = SunZe.multiply(pi.divide(180)).cos()
    sindSunZe = SunZe.multiply(pi.divide(180)).sin()

    SatZe = ee.Image.constant(img.get('MEAN_INCIDENCE_ZENITH_ANGLE_B5'))
    cosdSatZe = SatZe.multiply(pi.divide(180)).cos()
    sindSatZe = SatZe.multiply(pi.divide(180)).sin()

    SatAz = ee.Image.constant(img.get('MEAN_INCIDENCE_AZIMUTH_ANGLE_B5'))

    RelAz = SatAz.subtract(SunAz)
    cosdRelAz = RelAz.multiply(pi.divide(180)).cos()

    P = (ee.Image(101325)
         .multiply(
             ee.Image(1).subtract(
                 DEM.multiply(0.0000225577)
             ).pow(5.25588)
         )
         .multiply(0.01)
         .multiply(mask))

    Po = ee.Image(1013.25)

    ESUN = ee.Image(ee.Array([
        ee.Number(img.get('SOLAR_IRRADIANCE_B1')),
        ee.Number(img.get('SOLAR_IRRADIANCE_B2')),
        ee.Number(img.get('SOLAR_IRRADIANCE_B3')),
        ee.Number(img.get('SOLAR_IRRADIANCE_B4')),
        ee.Number(img.get('SOLAR_IRRADIANCE_B5')),
        ee.Number(img.get('SOLAR_IRRADIANCE_B6')),
        ee.Number(img.get('SOLAR_IRRADIANCE_B7')),
        ee.Number(img.get('SOLAR_IRRADIANCE_B8')),
        ee.Number(img.get('SOLAR_IRRADIANCE_B8A')),
        ee.Number(img.get('SOLAR_IRRADIANCE_B11')),
        ee.Number(img.get('SOLAR_IRRADIANCE_B2'))
    ])).toArray().toArray(1)

    ESUNImg = ESUN.arrayProject([0]).arrayFlatten([bands])

    imgArr = rescale.select(bands).toArray().toArray(1)

    Ltoa = imgArr.multiply(ESUN).multiply(cosdSunZe).divide(pi.multiply(d.pow(2)))

    bandCenter = ee.Image.constant(
        [443,490,560,665,705,740,783,842,865,1610,2190]
    ).divide(1000).toArray().toArray(1)

    Tr = (P.divide(Po)).multiply(
        ee.Image(0.008569).multiply(bandCenter.pow(-4))
    )

    denom = ee.Image(4).multiply(pi).multiply(cosdSatZe)
    Lr = ESUN.multiply(Tr).divide(denom)

    Lrc = Ltoa.subtract(Lr)

    pw = (Lrc.multiply(pi)
          .multiply(d.pow(2))
          .divide(ESUN.multiply(cosdSunZe)))

    Rrs = (pw.divide(pi)
           .arrayProject([0])
           .arrayFlatten([bands])
           .slice(0, 9))

    return Rrs.copyProperties(img, ['system:time_start'])


def chlorophyll(img):
    img = ee.Image(img)

    ndci = img.select('B5').subtract(img.select('B4')).divide(
        img.select('B5').add(img.select('B4'))
    )

    chl = ee.Image(14.039)\
        .add(ee.Image(86.115).multiply(ndci))\
        .add(ee.Image(194.325).multiply(ndci.pow(2)))

    return chl.updateMask(chl.lt(100))\
        .copyProperties(img, ['system:time_start'])


def secchi(img):
    img = ee.Image(img)

    ratio = img.select('B2').divide(img.select('B4')).max(0.0001)

    lnMOSD = ee.Image(1.4856).multiply(ratio.log()).add(0.2734)
    MOSD = ee.Image(10).pow(lnMOSD)
    sd = ee.Image(0.1777).multiply(MOSD).add(1.0813)

    return sd.updateMask(sd.lt(10))\
        .copyProperties(img, ['system:time_start'])


def trophicState(img):
    img = ee.Image(img)

    tsi = ee.Image(30.6).add(ee.Image(9.81).multiply(img.max(0.0001).log()))

    return tsi.updateMask(tsi.lt(200))\
        .copyProperties(img, ['system:time_start'])


def reclassify(img):
    img = ee.Image(img)

    tsiR = (img.lt(30).multiply(1)
        .add(img.gte(30).And(img.lt(40)).multiply(2))
        .add(img.gte(40).And(img.lt(50)).multiply(3))
        .add(img.gte(50).And(img.lt(60)).multiply(4))
        .add(img.gte(60).And(img.lt(70)).multiply(5))
        .add(img.gte(70).And(img.lt(80)).multiply(6))
        .add(img.gte(80).multiply(7)))

    return tsiR.copyProperties(img, ['system:time_start'])


# ---------------------------
# PIPELINE
# ---------------------------
Rrs_coll = FC.map(s2Correction)
chlor_a_coll = Rrs_coll.map(chlorophyll)
sd_coll = Rrs_coll.map(secchi)
tsi_coll = chlor_a_coll.map(trophicState)
tsi_collR = tsi_coll.map(reclassify)

# ---------------------------
# MAP
# ---------------------------
Map = geemap.Map(center=[10, 102], zoom=6)

Map.addLayer(chlor_a_coll.mean().clip(gulf_of_THA),
             {'min':0, 'max':40, 'palette':['blue','green','yellow','red']},
             'Chlorophyll')

Map.addLayer(sd_coll.mean().clip(gulf_of_THA),
             {'min':0, 'max':2},
             'Secchi')

Map.addLayer(tsi_coll.mean().clip(gulf_of_THA),
             {'min':30, 'max':80},
             'TSI')

Map.addLayer(tsi_collR.mode().clip(gulf_of_THA),
             {'min':1, 'max':7},
             'TSI Class')

Map

[Sentinel-2 Water Quality GEE Map](https://gustkt.github.io/GE338-Lab-5/proposal/Sentinel_2_Water_Quality.html)